# MoMADiff — Pilot-18 Canonical NPY Export Version

Purpose: generate the team-revised 18 Pilot prompts with MoMADiff, standardise each recovered motion to the shared Benchmark input contract, and export one canonical `.npy` per `prompt_id`.

**MoMADiff notebook:** Generate → Standardise → Validate → Export  
**Benchmark notebook:** Load → Validate → Route → Evaluate → Compare

# MoMADiff Pilot Evaluation — Clean Notebook

This notebook contains only the reusable pipeline for the pilot study:

1. Runtime & environment
2. MoMADiff setup and model loading
3. Motion generation
4. Existing metric evaluation — Individual Matching Score
5. Existing semantic evaluation — TMR++
6. Requirement-specific geometric evaluation
7. Human Gold validation

> **Note:** Debugging / repository-inspection / failed-attempt cells from the original notebook were removed. The geometric evaluator is still under refinement and is kept as a candidate evaluator rather than a finalized benchmark rule.

# 1. Runtime & Environment
Check the available GPU and runtime environment.

In [ ]:
!nvidia-smi

# 2. MoMADiff Setup
Clone the MoMADiff repository and install the required dependencies.

In [ ]:

!git clone https://github.com/zzysteve/MoMADiff.git

%cd /content/MoMADiff

!pip install git+https://github.com/openai/CLIP.git
!pip install -r requirements.txt

# 3. Pretrained Checkpoints
Download and verify the pretrained MoMADiff, KLVAE, and evaluation checkpoints.

In [ ]:
# Download & Setup Checkpoints


%cd /content/MoMADiff

# Download pretrained checkpoints
!python prepare/hf_download.py

# Create checkpoint directory
!mkdir -p /content/MoMADiff/checkpoints

# Copy HumanML3D (t2m) checkpoints to the location
# expected by MoMADiff
!cp -r /content/checkpoints_downloaded/t2m /content/MoMADiff/checkpoints/

print("\nCheckpoint setup complete!")

# Verify essential files
import os

required_files = [
    "./checkpoints/t2m/kl_vae_ver0-stable/args_for_pretrained_klvae.pkl",
    "./checkpoints/t2m/kl_vae_ver0-stable/net_last.pth",
    "./checkpoints/t2m/length_estimator/model/finest.tar",
    "./checkpoints/t2m/Trans-B_EMA_klv0-stable_masked-only_diff1000/model/latest_ema.tar"
]

for file in required_files:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗ MISSING:", file)

# 4. Imports & Model Components
Import the libraries and MoMADiff model components required for inference.

In [ ]:

%cd /content/MoMADiff

import os
import sys
import pickle
import itertools
import warnings
import imageio
import numpy as np

from os.path import join as pjoin
from argparse import ArgumentParser

import torch
import torch.nn.functional as F

from tqdm import tqdm
from torch.distributions.categorical import Categorical
from torch_ema import ExponentialMovingAverage

from models.klvae.autoencoder import AutoencoderKL as KLVAE
from models.mask_transformer.latent_transformer import MaskLatentTransformer
from models.length_est import LengthEstimator

from options.train_option import (
    add_train_mode_args,
    load_config_file_and_parse
)

from utils.diffusion import (
    create_model_and_diffusion,
    create_gaussian_diffusion_ddim
)

from utils.fixseed import fixseed
from utils.motion_process import recover_from_ric

from visualization.remove_fs import *
from visualization.plot_3d_global import plot_3d_motion

from utils.demo_util import (
    filtering,
    sanitize_filename
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="matplotlib.animation"
)

clip_version = "ViT-B/32"

print("Imports OK!")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 5. Model Configuration
Configure MoMADiff for HumanML3D text-to-motion generation.

In [ ]:

sys.argv = [
    'notebook',
    '--name', 'Trans-B_EMA_klv0-stable_masked-only_diff1000',
    '--config', 'configs/t2m.yaml',
    '--use_ema', 'true',
    '--diffusion_steps', '1000',
    '--loss_strategy', 'masked',
    '--ddim_steps', '100',
    '--trans_infer_timesteps', '9',
    '--guidance_param', '3'
]

# Load configuration
parser = ArgumentParser()
parser = add_train_mode_args(parser)
opt = load_config_file_and_parse(parser)

# Fix random seed
fixseed(opt.seed)

# Set device
opt.device = torch.device(
    "cpu" if opt.device == -1
    else "cuda:" + str(opt.device)
)

# Model directories
root_dir = pjoin(
    opt.checkpoints_dir,
    opt.dataset_name,
    opt.name
)

model_dir = pjoin(
    root_dir,
    "model"
)

# ==========================================
# Generation / Inference settings
# ==========================================

opt.time_steps = 7
opt.cond_scale = 3

# ==========================================
# Check configuration
# ==========================================

print("Configuration loaded!")
print("----------------------------")
print("Device:", opt.device)
print("Dataset:", opt.dataset_name)
print("Checkpoint directory:", opt.checkpoints_dir)
print("Model directory:", model_dir)
print("Time steps:", opt.time_steps)
print("Condition scale:", opt.cond_scale)

# 6. Model Loading Functions
Define functions for loading the KLVAE, MoMADiff model, and motion length estimator.

In [ ]:

def load_kl_model(opt):

    kl_args_path = pjoin(
        opt.checkpoints_dir,
        opt.dataset_name,
        opt.kl_name,
        "args_for_pretrained_klvae.pkl"
    )

    with open(kl_args_path, "rb") as f:
        args = pickle.load(f)

    kl_model = KLVAE(
        args,
        args.output_emb_width,
        args.down_t,
        args.stride_t,
        args.width,
        args.depth,
        args.dilation_growth_rate
    )

    args.klvae_pth = pjoin(
        opt.checkpoints_dir,
        opt.dataset_name,
        opt.kl_name,
        "net_last.pth"
    )

    ckpt = torch.load(
        args.klvae_pth,
        map_location="cpu"
    )

    kl_model.load_state_dict(
        ckpt["net"],
        strict=True
    )

    print(f"Loading KL Model: {opt.kl_name}")

    return kl_model, args

In [ ]:
def load_models(model_opt, which_model):

    cond_mode = (
        "text"
        if not model_opt.unconstrained
        else "uncond"
    )

    diff_model, diffusion = create_model_and_diffusion(
        model_opt
    )

    latent_transformer = MaskLatentTransformer(
        code_dim=model_opt.code_dim,
        cond_mode=cond_mode,
        latent_dim=model_opt.latent_dim,
        ff_size=model_opt.ff_size,
        num_layers=model_opt.n_layers,
        num_heads=model_opt.n_heads,
        dropout=model_opt.dropout,
        clip_dim=512,
        cond_drop_prob=model_opt.cond_drop_prob,
        opt=model_opt,
        clip_version=clip_version,
        use_ema=model_opt.use_ema
    )

    ckpt_path = pjoin(
        model_opt.checkpoints_dir,
        model_opt.dataset_name,
        model_opt.name,
        "model",
        which_model
    )

    ckpt = torch.load(
        ckpt_path,
        map_location=model_opt.device
    )

    model_key = (
        "t2m_transformer"
        if "t2m_transformer" in ckpt
        else "trans"
    )

    if "ema" in which_model:

        print("[Info] Using EMA")

        ema = ExponentialMovingAverage(
            itertools.chain(
                latent_transformer.parameters(),
                diff_model.parameters()
            ),
            decay=model_opt.ema_decay
        )

        ema.load_state_dict(ckpt)

        ema.copy_to(
            itertools.chain(
                latent_transformer.parameters(),
                diff_model.parameters()
            )
        )

        print("Loading EMA weights")

    else:

        missing_keys, unexpected_keys = (
            latent_transformer.load_state_dict(
                ckpt[model_key],
                strict=False
            )
        )

        assert len(unexpected_keys) == 0

        assert all(
            k.startswith("clip_model.")
            for k in missing_keys
        )

        diff_model.load_state_dict(
            ckpt["diff_model"]
        )

    return latent_transformer, diff_model, diffusion

In [ ]:
def load_len_estimator(opt):

    model = LengthEstimator(
        512,
        50
    )

    ckpt_path = pjoin(
        opt.checkpoints_dir,
        opt.dataset_name,
        "length_estimator",
        "model",
        "finest.tar"
    )

    ckpt = torch.load(
        ckpt_path,
        map_location=opt.device
    )

    model.load_state_dict(
        ckpt["estimator"]
    )

    print(
        f'Loading Length Estimator '
        f'from epoch {ckpt["epoch"]}!'
    )

    return model


print("Model loading functions ready!")

# 7. Load Pretrained Models
Load the pretrained KLVAE, MoMADiff, and motion length estimator.

In [ ]:

# KLVAE
# -------------------------

encdec_model, encdec_opt = load_kl_model(opt)

encdec_model = encdec_model.to(
    opt.device
)

encdec_model.eval()

print("KLVAE loaded!")


# -------------------------
# MoMADiff
# -------------------------

checkpoint_file = "latest_ema.tar"

latent_transformer, diff_model, diffusion = (
    load_models(
        opt,
        checkpoint_file
    )
)

latent_transformer = latent_transformer.to(
    opt.device
)

diff_model = diff_model.to(
    opt.device
)

latent_transformer.eval()
diff_model.eval()


# -------------------------
# DDIM
# -------------------------

if opt.ddim_steps != -1:

    print(
        f"Using DDIM with "
        f"{opt.ddim_steps} steps"
    )

    diffusion = create_gaussian_diffusion_ddim(
        opt,
        opt.ddim_steps
    )


# -------------------------
# Length estimator
# -------------------------

length_estimator = load_len_estimator(opt)

length_estimator = length_estimator.to(
    opt.device
)

length_estimator.eval()


# -------------------------
# Motion information
# -------------------------

opt.nb_joints = (
    21
    if opt.dataset_name == "kit"
    else 22
)

mean = np.load(
    pjoin(
        opt.meta_dir,
        "mean.npy"
    )
)

std = np.load(
    pjoin(
        opt.meta_dir,
        "std.npy"
    )
)


def inv_transform(data):
    return data * std + mean


print("")
print("============================")
print("MoMADiff READY!")
print("============================")
print("Device:", opt.device)
print("Dataset:", opt.dataset_name)

In [ ]:
# ============================================================
# CLEAN START — delete previously generated MoMADiff motions
# ============================================================
# Run this cell ONCE before the new test generation.
# It deletes local generated motion outputs under /content/MoMADiff/generation/test_gif.
# It does NOT delete Google Drive Pilot evidence.

import os
import shutil

GENERATED_MOTION_DIR = "/content/MoMADiff/generation/test_gif"

if os.path.isdir(GENERATED_MOTION_DIR):
    shutil.rmtree(GENERATED_MOTION_DIR)
    print("Deleted old generated motions:", GENERATED_MOTION_DIR)
else:
    print("No old generated-motion folder found:", GENERATED_MOTION_DIR)

os.makedirs(GENERATED_MOTION_DIR, exist_ok=True)
print("Created clean output folder:", GENERATED_MOTION_DIR)


# 8. Common Visualization Standard

Define the project-wide visualization before motion generation. Every motion generated by `generate_motion()` is automatically rendered with this standard.

- Stickman style: MoMADiff / MoLingo style
- Initial root XZ = `(0, 0)`
- Y-up
- +Z = Forward / -Z = Backward
- +X = Right / -X = Left
- Ground plane = `Y = 0`
- Fixed 3/4 camera: `elev = 30°`, `azim = -45°`
- Root trajectory displayed
- Grid / axes displayed


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image
from utils.paramUtil import t2m_kinematic_chain

# Match the shared benchmark_utils.py visual convention.
DEFAULT_ELEV = 30
DEFAULT_AZIM = -45


def render_standardized_motion(motion_path, title, output_path=None, fps=20,
                               display_gif=True,
                               elev=DEFAULT_ELEV, azim=DEFAULT_AZIM):
    """Render MoMADiff in the same benchmark style as the member's MotionHiFlow renderer."""
    motion = np.load(motion_path).copy()
    if motion.ndim != 3 or motion.shape[1:] != (22, 3):
        raise ValueError(f"Expected motion shape (T, 22, 3), got {motion.shape}")

    # 1) Ground alignment: lowest point -> Y = 0.
    motion[:, :, 1] -= motion[:, :, 1].min()

    # 2) Origin alignment: first-frame root XZ -> (0, 0).
    motion[:, :, 0] -= motion[0, 0, 0]
    motion[:, :, 2] -= motion[0, 0, 2]

    # 3) Benchmark axis convention: +X = Right, -X = Left.
    motion[:, :, 0] *= -1.0

    trajectory = motion[:, 0, :]

    if output_path is None:
        output_path = "/content/standardized_motion.gif"

    # User-requested fixed X display range/ticks.
    x_min, x_max = -1.0, 2.5
    x_ticks = np.arange(-1.0, 2.5 + 0.001, 0.5)

    # Match benchmark_utils scene logic for Z/Y.
    margin = 1.0
    min_span = 3.0
    z_min = float(trajectory[:, 2].min()) - margin
    z_max = float(trajectory[:, 2].max()) + margin
    if z_max - z_min < min_span:
        centre = (z_min + z_max) / 2
        z_min, z_max = centre - min_span / 2, centre + min_span / 2
    y_min = 0.0
    y_max = float(motion[:, :, 1].max()) + 0.25

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection="3d")

    # Same ground wireframe style as benchmark_utils.py.
    GX, GZ = np.meshgrid(np.linspace(x_min, x_max, 10),
                         np.linspace(z_min, z_max, 10))
    GY = np.zeros_like(GX)

    # Same direction arrows/labels as member renderer.
    L = 0.8
    AXES = [
        (( L, 0, 0), "+X RIGHT"),
        ((-L, 0, 0), "-X LEFT"),
        ((0,  L, 0), "+Z FORWARD"),
        ((0, -L, 0), "-Z BACKWARD"),
        ((0, 0,  L), "+Y UP"),
    ]

    def update(frame):
        ax.cla()
        joints = motion[frame]

        # motion X -> plot X, motion Z -> plot Y, motion Y -> plot Z.
        for chain in t2m_kinematic_chain:
            p = joints[chain]
            ax.plot(p[:, 0], p[:, 2], p[:, 1], linewidth=2)
        ax.scatter(joints[:, 0], joints[:, 2], joints[:, 1], s=10)

        # Root trajectory on the ground.
        root = motion[:frame + 1, 0, :]
        ax.plot(root[:, 0], root[:, 2], np.zeros(len(root)),
                linestyle="--", linewidth=1.5)

        ax.plot_wireframe(GX, GZ, GY, linewidth=0.3, alpha=0.35)

        for (dx, dy, dz), label in AXES:
            ax.quiver(0, 0, 0, dx, dy, dz, arrow_length_ratio=0.12)
            ax.text(dx, dy, dz, label)

        ax.set_xlim(x_min, x_max)
        ax.set_xticks(x_ticks)
        ax.set_ylim(z_min, z_max)
        ax.set_zlim(y_min, y_max)
        ax.set_box_aspect((1, 1, 0.7))
        ax.view_init(elev=elev, azim=azim)

        ax.set_xlabel("X (+Right / -Left)")
        ax.set_ylabel("Z (+Forward / -Backward)")
        ax.set_zlabel("Y (+Up)")
        ax.grid(True)
        ax.set_title(f"{title}\nFrame {frame + 1}/{len(motion)}")

    ani = FuncAnimation(fig, update, frames=len(motion), interval=1000 / fps)
    ani.save(str(output_path), writer=PillowWriter(fps=fps), dpi=100)
    plt.close(fig)

    print("Motion shape:", motion.shape)
    print("Initial root XZ:", trajectory[0, [0, 2]])
    print("Final root XZ:", trajectory[-1, [0, 2]])
    print("Total XZ displacement:", trajectory[-1, [0, 2]] - trajectory[0, [0, 2]])
    print("X display range: -1.0 to +2.5 (0.5 ticks)")
    print("Saved:", output_path)
    if display_gif:
        display(Image(filename=output_path))
    return output_path


# 9. Motion Generation
Define `generate_motion()` after the common renderer so every newly generated motion is automatically saved and displayed using the Common Visualization Standard.


In [ ]:
# ==========================================
# Cell 8 — Motion Generation Function
# ==========================================

def generate_motion(prompt, length=0, repeat_times=1, fps=20):
    """
    Generate a motion from a text prompt using MoMADiff.

    Parameters
    ----------
    prompt : str
        Text description of the motion.

    length : int
        Motion token length.
        0 = automatically estimate the motion length.

    repeat_times : int
        Number of motions to generate.

    fps : int
        FPS of the generated GIF.
    """

    # Create a safe filename from the prompt
    motion_name = sanitize_filename(prompt)

    result_dir = pjoin(
        "./generation/test_gif",
        motion_name
    )
    os.makedirs(result_dir, exist_ok=True)

    # ------------------------------------------
    # 1. Estimate motion length
    # ------------------------------------------

    if length == 0:
        print("Estimating motion length...")

        with torch.no_grad():
            text_embedding = latent_transformer.encode_text(prompt)

            pred_dis = length_estimator(text_embedding)

            probs = F.softmax(
                pred_dis,
                dim=-1
            )

            token_lens = Categorical(probs).sample()
            token_lens = token_lens[:1]

    else:
        token_lens = torch.tensor(
            [length],
            device=opt.device,
            dtype=torch.long
        )

    # Each token corresponds to 4 motion frames
    m_length = token_lens * 4

    print()
    print("============================")
    print("Prompt:", prompt)
    print("Motion length:", m_length.item(), "frames")
    print("============================")
    print()

    gif_paths = []
    npy_paths = []

    # ------------------------------------------
    # 2. Generate motion
    # ------------------------------------------

    for r in range(repeat_times):

        print(f"Generating motion {r + 1}/{repeat_times}...")

        with torch.no_grad():

            pred_latent, gen_steps = latent_transformer.generate(
                [prompt],
                token_lens,
                opt.time_steps,
                opt.cond_scale,
                diffusion=diffusion,
                diff_model=diff_model,
                output_inference_step=True,
                noise_schedule=None
            )

            # Decode latent representation
            pred_motions = encdec_model.decode(pred_latent)

            pred_motions = (
                pred_motions
                .detach()
                .cpu()
                .numpy()
            )

            # De-normalize motion
            data = inv_transform(pred_motions)

            data = data[:, :m_length.item()]

            # Recover 3D joint positions
            joint = recover_from_ric(
                torch.from_numpy(data).float(),
                22
            ).numpy()

            joint = joint[0]

            # Smooth motion
            joint = filtering(
                joint,
                m_length,
                window_size=15,
                poly_order=3
            )

            # ------------------------------------------
            # 3. Save NPY
            # ------------------------------------------

            npy_path = pjoin(
                result_dir,
                f"{motion_name}_repeat{r}_len{m_length.item()}.npy"
            )

            np.save(npy_path, joint)

            # ------------------------------------------
            # 4. Create STANDARDIZED GIF automatically
            # ------------------------------------------

            gif_path = pjoin(
                result_dir,
                f"{motion_name}_repeat{r}_standardized.gif"
            )

            render_standardized_motion(
                motion_path=npy_path,
                title=prompt,
                output_path=gif_path,
                fps=fps,
                display_gif=False
            )

            npy_paths.append(npy_path)
            gif_paths.append(gif_path)

            print("NPY saved:", npy_path)
            print("Standardized GIF saved:", gif_path)
            print("Joint shape:", joint.shape)

    # ------------------------------------------
    # 5. Display generated GIF
    # ------------------------------------------

    from IPython.display import Image, display

    print()

    for path in gif_paths:
        display(
            Image(filename=path)
        )

    print()
    print("Motion generation complete!")

    return {"npy_paths": npy_paths, "gif_paths": gif_paths}


print("generate_motion() ready!")

# 9. Pilot-18 Generation → Canonical Benchmark `.npy` Export

This section implements the agreed separation of responsibilities:

**MoMADiff notebook = Generate + Standardise + Export**  
**Benchmark notebook = Load + Validate + Evaluate + Compare**

The Benchmark notebook does **not** load MoMADiff or call a MoMADiff runner. It receives one canonical `.npy` per Pilot prompt.

## Canonical benchmark motion contract
Every exported file must satisfy:
- shape: `[T, 22, 3]`
- coordinates: global XYZ
- `+Y`: up
- `+Z`: forward
- `+X`: right
- units: metres
- FPS: 20
- initial root XZ: `(0, 0)`
- ground aligned to `Y = 0`

The exact 18 Pilot prompts and target frame counts come from the uploaded Pilot definition.

In [ ]:
# 9.1 Load the 18-prompt Pilot definition from JSON
import json
from pathlib import Path
from google.colab import files

# Upload pilot_benchmark_definition(3).json when prompted
uploaded = files.upload()
json_files = [name for name in uploaded if name.lower().endswith(".json")]
if not json_files:
    raise FileNotFoundError("Please upload the Pilot benchmark definition JSON file.")

PILOT_JSON_PATH = Path(json_files[0])
with PILOT_JSON_PATH.open("r", encoding="utf-8") as f:
    PILOT_DEFINITION = json.load(f)

PILOT_PROMPTS = PILOT_DEFINITION["prompts"]
assert len(PILOT_PROMPTS) == 18, f"Expected 18 Pilot prompts, got {len(PILOT_PROMPTS)}"

print("Loaded:", PILOT_JSON_PATH.name)
print("Pilot prompts:", len(PILOT_PROMPTS))
for p in PILOT_PROMPTS:
    print(f'{p["prompt_id"]:6s} | {p["capability"]} | {p["difficulty"]:6s} | {p["target_frames_20fps"]:3d} frames | {p["text"]}')


In [ ]:
# 9.2 Canonical MoMADiff -> Benchmark standardisation
import numpy as np
BENCHMARK_FPS = 20
BENCHMARK_JOINTS = 22

def standardize_for_benchmark(raw_xyz):
    motion = np.asarray(raw_xyz, dtype=np.float32).copy()
    if motion.ndim != 3 or motion.shape[1:] != (22, 3):
        raise ValueError(f"Expected [T,22,3], got {motion.shape}")
    if not np.isfinite(motion).all():
        raise ValueError("Motion contains NaN or infinite values.")
    motion[:, :, 1] -= motion[:, :, 1].min()   # ground -> Y=0
    motion[:, :, 0] -= motion[0, 0, 0]         # root X -> 0
    motion[:, :, 2] -= motion[0, 0, 2]         # root Z -> 0
    motion[:, :, 0] *= -1.0                    # +X = Right
    return motion

def validate_canonical_motion(motion, expected_frames=None):
    errors = []
    if not isinstance(motion, np.ndarray):
        return [f"not NumPy: {type(motion)}"]
    if motion.ndim != 3 or motion.shape[1:] != (22, 3):
        errors.append(f"shape={motion.shape}, expected [T,22,3]")
        return errors
    if not np.isfinite(motion).all(): errors.append("contains NaN/Inf")
    if expected_frames is not None and motion.shape[0] != expected_frames:
        errors.append(f"frames={motion.shape[0]}, expected={expected_frames}")
    if not np.allclose(motion[0,0,[0,2]], 0.0, atol=1e-5): errors.append("initial root XZ is not origin")
    if abs(float(motion[:,:,1].min())) > 1e-5: errors.append("ground min Y != 0")
    return errors
print("Canonical standardisation + validation ready.")

In [ ]:
# 9.3 Output folder handed to Benchmark.ipynb
from pathlib import Path
from datetime import datetime, timezone
EXPORT_ROOT = Path("/content/benchmark_inputs/MoMADiff")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print("Benchmark export folder:", EXPORT_ROOT)

In [ ]:
# 9.4 Generate ONE Pilot prompt and export canonical .npy

def generate_and_export_pilot_prompt(prompt_def, overwrite=False):
    prompt_id = prompt_def["prompt_id"]
    text = prompt_def["text"]
    target_frames = int(prompt_def["target_frames_20fps"])
    if target_frames % 4 != 0:
        raise ValueError(f"{prompt_id}: target frames must be divisible by 4 for MoMADiff")
    token_length = target_frames // 4
    out_path = EXPORT_ROOT / f"{prompt_id}.npy"
    meta_path = EXPORT_ROOT / f"{prompt_id}.json"
    if out_path.exists() and not overwrite:
        print(f"SKIP {prompt_id}: already exists -> {out_path}")
        return {"prompt_id": prompt_id, "status": "SKIP", "path": str(out_path)}

    generated = generate_motion(prompt=text, length=token_length, repeat_times=1, fps=BENCHMARK_FPS)
    raw_path = generated["npy_paths"][0]
    raw_xyz = np.load(raw_path)
    canonical = standardize_for_benchmark(raw_xyz)
    errors = validate_canonical_motion(canonical, expected_frames=target_frames)
    if errors:
        raise ValueError(f"{prompt_id} canonical validation failed: {errors}")
    np.save(out_path, canonical.astype(np.float32))

    metadata = {
        "model":"MoMADiff", "prompt_id":prompt_id, "prompt":text,
        "capability":prompt_def["capability"], "difficulty":prompt_def["difficulty"],
        "target_frames":target_frames, "actual_frames":int(canonical.shape[0]),
        "fps":20, "shape":list(canonical.shape), "coordinate_system":"global_xyz",
        "up_axis":"+Y", "forward_axis":"+Z", "right_axis":"+X", "units":"metres",
        "source_raw_npy":str(raw_path), "benchmark_npy":str(out_path),
        "created_utc":datetime.now(timezone.utc).isoformat()
    }
    with open(meta_path,"w",encoding="utf-8") as f: json.dump(metadata,f,indent=2,ensure_ascii=False)
    print(f"PASS {prompt_id} -> {out_path} | shape={canonical.shape}")
    return {"prompt_id":prompt_id,"status":"PASS","path":str(out_path),"metadata":metadata}
print("Single-prompt export function ready.")

## 9.5 Smoke test
Run one prompt first. It should export `C1-01.npy` with shape `(100, 22, 3)`.

In [ ]:
smoke = generate_and_export_pilot_prompt(PILOT_PROMPTS[0], overwrite=False)
print(smoke)

## 9.6 Generate all 18 Pilot motions
Existing canonical files are skipped by default, so an interrupted Colab session can resume.

In [ ]:
results=[]
print("="*72); print("MoMADiff Pilot-18 -> Canonical Benchmark NPY Export"); print("="*72)
for i,p in enumerate(PILOT_PROMPTS,1):
    print("\n"+"-"*72)
    print(f"[{i}/18] {p['prompt_id']} | {p['difficulty']} | {p['target_frames_20fps']} frames")
    print(p["text"])
    try:
        r=generate_and_export_pilot_prompt(p, overwrite=False)
    except Exception as e:
        r={"prompt_id":p["prompt_id"],"status":"FAIL","error":f"{type(e).__name__}: {e}"}
        print("FAIL:",r["error"])
    results.append(r)
print("\nSUMMARY")
for r in results: print(f"{r['prompt_id']:6s} : {r['status']}")

In [ ]:
# 9.7 Final hand-off validation + manifest
manifest={"model":"MoMADiff","benchmark_subset":"Pilot","prompt_count_expected":18,
          "input_contract":{"shape":"[T,22,3]","coordinates":"global XYZ","up_axis":"+Y",
          "forward_axis":"+Z","right_axis":"+X","units":"metres","fps":20,
          "initial_root_xz":[0.0,0.0],"ground_y":0.0},"files":[]}
all_ok=True
for p in PILOT_PROMPTS:
    path=EXPORT_ROOT/f"{p['prompt_id']}.npy"
    row={"prompt_id":p["prompt_id"],"prompt":p["text"],"target_frames":p["target_frames_20fps"],"filename":path.name,"exists":path.exists()}
    if path.exists():
        motion=np.load(path); errors=validate_canonical_motion(motion,p["target_frames_20fps"])
        row.update({"shape":list(motion.shape),"valid":len(errors)==0,"errors":errors})
    else:
        row.update({"valid":False,"errors":["missing file"]})
    all_ok &= row["valid"]; manifest["files"].append(row)
manifest["all_18_ready"]=bool(all_ok)
manifest_path=EXPORT_ROOT/"manifest.json"
with open(manifest_path,"w",encoding="utf-8") as f: json.dump(manifest,f,indent=2,ensure_ascii=False)
print("All 18 ready for Benchmark.ipynb:",all_ok)
print("Manifest:",manifest_path)
for row in manifest["files"]:
    print(f"{'PASS' if row['valid'] else 'FAIL':4s} | {row['prompt_id']:6s} | {row.get('shape')} | {row['filename']}")

## 9.8 Benchmark hand-off
The Benchmark receives only the canonical files:

```text
/content/benchmark_inputs/MoMADiff/
├── C1-01.npy
├── C1-02.npy
├── ...
├── C5-16.npy
└── manifest.json
```

Raw MoMADiff generation files and GIFs remain model-side/debugging artifacts. They are not Benchmark inputs.

# 10. Existing Metric Evaluation

## 10.1 Individual Matching Score

Use the pretrained HumanML3D text-motion evaluator to calculate an individual text–motion distance. **Lower = better global text-motion alignment.**

Pilot purpose: test whether a global semantic metric can detect requirement-level failures such as **Order**, **Count**, and **Action Omission**.

In [ ]:
# ========================================
# Download GloVe for HumanML3D Evaluator
# ========================================

%cd /content/MoMADiff

!rm -rf glove

!gdown --fuzzy \
"https://drive.google.com/file/d/1cmXKUT31pqd7_XpJAiWEo1K81TMYHA5n/view?usp=sharing"

!unzip -oq glove.zip
!rm -f glove.zip

print("GloVe setup complete ✅")

In [ ]:
# ========================================
# 9.1.1 Matching Score Evaluator Setup
# ========================================

%cd /content/MoMADiff

import os
import numpy as np
import torch
import torch.nn.functional as F
import imageio

from types import SimpleNamespace
from torch.distributions.categorical import Categorical
from utils.word_vectorizer import WordVectorizer
from models.klvae.evaluator_wrapper import EvaluatorModelWrapper
from utils.metrics import euclidean_distance_matrix
from utils.motion_process import recover_from_ric
from visualization.plot_3d_global import plot_3d_motion
from IPython.display import Image, display

w_vectorizer = WordVectorizer('./glove', 'our_vab')

eval_opt = SimpleNamespace(
    dataset_name='t2m',
    checkpoints_dir='./checkpoints',
    device=opt.device,
    dim_movement_enc_hidden=512,
    dim_movement_latent=512,
    unit_length=4,
)

individual_eval_wrapper = EvaluatorModelWrapper(
    eval_opt,
    w_vectorizer=w_vectorizer
)

print("Individual Matching Score evaluator ready ✅")

In [ ]:
%cd /content/MoMADiff

import os
import numpy as np
import torch
import torch.nn.functional as F
import imageio

from torch.distributions.categorical import Categorical
from utils.motion_process import recover_from_ric
from visualization.plot_3d_global import plot_3d_motion
from utils.metrics import euclidean_distance_matrix
from IPython.display import Image, display


def run_pilot_metric(
    prompt,
    motion_name,
    save_dir="/content/MoMADiff/pilot_metric_test",
    fps=20
):
    """
    Generate one MoMADiff motion and evaluate the SAME generation.

    Outputs:
    - normalized 263-D evaluator motion (.npy)
    - 22-joint visual motion (.npy)
    - GIF
    - Individual Matching Score

    Returns:
    dict containing prompt, motion length, matching score,
    and saved file paths.
    """

    os.makedirs(save_dir, exist_ok=True)

    print("=" * 75)
    print("Prompt:", prompt)
    print("=" * 75)

    # --------------------------------------------------
    # 1. Estimate motion length
    # --------------------------------------------------
    with torch.no_grad():
        text_embedding_clip = latent_transformer.encode_text(prompt)

        pred_dis = length_estimator(text_embedding_clip)
        probs = F.softmax(pred_dis, dim=-1)

        token_lens = Categorical(probs).sample()
        token_lens = token_lens[:1]

        m_length = token_lens * 4

    # --------------------------------------------------
    # 2. Generate ONE motion
    # --------------------------------------------------
    with torch.no_grad():
        pred_latent, _ = latent_transformer.generate(
            [prompt],
            token_lens,
            opt.time_steps,
            opt.cond_scale,
            diffusion=diffusion,
            diff_model=diff_model,
            output_inference_step=True,
            noise_schedule=None
        )

        # Normalized HumanML3D 263-D representation
        pred_motions_eval = encdec_model.decode(pred_latent)

    # --------------------------------------------------
    # 3. Save evaluator representation
    # --------------------------------------------------
    eval_motion = (
        pred_motions_eval[0]
        .detach()
        .cpu()
        .numpy()
    )

    eval_path = os.path.join(
        save_dir,
        f"{motion_name}_eval.npy"
    )

    np.save(eval_path, eval_motion)

    # --------------------------------------------------
    # 4. Convert SAME generation to 22-joint motion
    # --------------------------------------------------
    visual_data = inv_transform(
        pred_motions_eval.detach().cpu().numpy()
    )

    visual_data = visual_data[:, :m_length.item()]

    joint = recover_from_ric(
        torch.from_numpy(visual_data).float(),
        22
    ).numpy()[0]

    visual_path = os.path.join(
        save_dir,
        f"{motion_name}_visual.npy"
    )

    np.save(visual_path, joint)

    # --------------------------------------------------
    # 5. Create GIF
    # --------------------------------------------------
    frames = plot_3d_motion(
        [joint, None, None]
    )

    gif_path = os.path.join(
        save_dir,
        f"{motion_name}.gif"
    )

    imageio.mimsave(
        gif_path,
        frames.cpu().numpy(),
        fps=fps,
        loop=0
    )

    # --------------------------------------------------
    # 6. Prepare evaluator text input
    # --------------------------------------------------
    word_emb, pos_ohot = (
        individual_eval_wrapper.get_text_embedding(prompt)
    )

    word_emb = word_emb.unsqueeze(0)
    pos_ohot = pos_ohot.unsqueeze(0)

    tokens = [
        x.text + "/" + x.pos_
        for x in individual_eval_wrapper.pos_processor(prompt)
    ]

    tokens = [
        w
        for w in tokens
        if w.split("/")[1] != "PUNCT"
    ]

    sent_len_value = (
        min(
            len(tokens),
            individual_eval_wrapper.opt.max_text_len
        )
        + 2
    )

    sent_len = torch.tensor(
        [sent_len_value],
        device=opt.device
    )

    m_lens = torch.tensor(
        [pred_motions_eval.shape[1]],
        device=opt.device
    )

    # --------------------------------------------------
    # 7. Calculate Individual Matching Score
    # --------------------------------------------------
    with torch.no_grad():
        et_pred, em_pred = (
            individual_eval_wrapper.get_co_embeddings(
                word_emb,
                pos_ohot,
                sent_len,
                pred_motions_eval,
                m_lens
            )
        )

    matching_score = euclidean_distance_matrix(
        et_pred.detach().cpu().numpy(),
        em_pred.detach().cpu().numpy()
    )[0, 0]

    matching_score = float(matching_score)

    # --------------------------------------------------
    # 8. Display results
    # --------------------------------------------------
    print("\n=== PILOT METRIC RESULT ===")
    print("Motion length :", int(m_lens.item()))
    print("Matching Score:", round(matching_score, 4))

    print("\nSaved files:")
    print("Evaluator:", eval_path)
    print("Visual   :", visual_path)
    print("GIF      :", gif_path)

    print("\n=== MOTION VISUAL ===")
    display(Image(filename=gif_path))

    # --------------------------------------------------
    # 9. Return structured result
    # --------------------------------------------------
    return {
        "Prompt": prompt,
        "Motion Name": motion_name,
        "Motion Length": int(m_lens.item()),
        "Matching Score": matching_score,
        "Eval Path": eval_path,
        "Visual Path": visual_path,
        "GIF Path": gif_path
    }


print("run_pilot_metric() ready!")

### 10.1.2 Example — Order Failure Case

Generate and evaluate the same motion with the HumanML3D Matching Score.

In [ ]:
result_walk_turn = run_pilot_metric(
    "A person walks forward and makes a right turn.",
    "temporal_walk_turn"
)

## 10.2 TMR++ Semantic Evaluation

Use pretrained TMR++ as a second existing semantic evaluator. MoMADiff `(T, 22, 3)` joint motions are converted to GuoH3D features before evaluation.

This section removes the dependency/debugging cells from the original notebook and keeps only the successful setup path.

In [ ]:
# ========================================
# 9.2.1 Install TMR++ and Dependencies
# ========================================

%cd /content

!rm -rf TMRPlusPlus
!git clone https://github.com/leorebensabath/TMRPlusPlus.git

!pip install -q hydra-core omegaconf colorlog pytorch-lightning gdown

print("TMR++ setup complete ✅")

In [ ]:
# ========================================
# 9.2.2 Download Pretrained TMR++ Model
# ========================================

%cd /content

!rm -rf /content/tmr_babel_guoh3dfeats

!gdown --folder \
"https://drive.google.com/drive/folders/1IU-XTSjY3zfqwbxiH2K4TfX1j70zv28B" \
-O /content/tmr_babel_guoh3dfeats

print("TMR++ pretrained model downloaded ✅")

In [ ]:
# ========================================
# 9.2.3 Prepare TMR++ for Arbitrary Pilot Text
# ========================================

import json
import shutil
import os

src = "/content/tmr_babel_guoh3dfeats"
dst = "/content/tmr_babel_test"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

config_path = os.path.join(dst, "config.json")

with open(config_path, "r") as f:
    cfg = json.load(f)

# Disable precomputed BABEL token embeddings so new pilot text can be encoded.
cfg["data"]["text_to_token_emb"]["preload"] = False

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("TMR++ individual-text setup ready ✅")
print("Token preload:", cfg["data"]["text_to_token_emb"]["preload"])

In [ ]:
# ========================================
# 9.2.4 MoMADiff -> GuoH3D Conversion
# ========================================

import sys
import numpy as np

sys.path.insert(0, "/content/TMRPlusPlus")
from src.guofeats import joints_to_guofeats

def convert_to_guoh3d(motion_path, output_path):
    """Convert MoMADiff (T,22,3) joint motion to TMR++ GuoH3D features."""
    motion = np.load(motion_path)

    if motion.ndim != 3 or motion.shape[1:] != (22, 3):
        raise ValueError(f"Expected (T,22,3), got {motion.shape}")

    features = joints_to_guofeats(motion.copy())
    np.save(output_path, features)

    print("=== GuoH3D Conversion ===")
    print("Input :", motion.shape)
    print("Output:", features.shape)
    print("NaN   :", np.isnan(features).any())
    print("Inf   :", np.isinf(features).any())
    print("Saved :", output_path)

    return output_path

print("convert_to_guoh3d() ready ✅")

### 10.2.5 Example — WALK → TURN

Convert the generated `WALK → TURN` motion and calculate its TMR++ similarity for the full prompt.

In [ ]:
import glob
import os

pattern = (
    "/content/MoMADiff/generation/test_gif/"
    "a_person_walks_forward_and_makes_a_right_turn/*.npy"
)

files = glob.glob(pattern)
if not files:
    raise FileNotFoundError(
        "WALK → TURN motion not found. Run the generation cell first."
    )

motion_path = max(files, key=os.path.getmtime)
tmr_motion_path = "/content/walk_turn_guoh3d.npy"

print("Selected:", motion_path)
convert_to_guoh3d(motion_path, tmr_motion_path)

In [ ]:
%cd /content/TMRPlusPlus

!python text_motion_sim.py \
    run_dir=/content/tmr_babel_test \
    text="A person walks forward and makes a right turn." \
    npy=/content/walk_turn_guoh3d.npy

### 10.2.6 Optional Contrastive TMR++ Test

Keep the same motion fixed and change only the text. This is useful for testing whether TMR++ is sensitive to an atomic requirement.

In [ ]:
#A person walks forward and makes a right turn. のTMRを確認
%%bash
cd /content/TMRPlusPlus

texts=(
    "A person walks forward and makes a right turn."
    "A person walks forward."
    "A person turns right."
)

for text in "${texts[@]}"; do
    echo ""
    echo "=================================================="
    echo "TEXT: $text"
    echo "=================================================="

    python text_motion_sim.py \
        run_dir=/content/tmr_babel_test \
        text="$text" \
        npy=/content/walk_turn_guoh3d.npy
done

# 11. Requirement-Specific Geometric Evaluation

Existing semantic metrics measure global text-motion alignment. Candidate geometric / kinematic evaluators instead measure the motion evidence needed for each atomic requirement.

Current priority in this notebook:
- `Order: A → B → C`
- `Count: N`
- `Action: X` / Action Omission

> These rules must be validated against Human Gold Labels before being frozen for the main benchmark.

## 11.1 Candidate Evaluator — Order: WALK → TURN

Current candidate features:
- root trajectory
- body heading
- turn onset
- pre-turn movement
- temporal phase separation

The code below preserves the current pilot analysis. It is **not yet a finalized PASS/FAIL threshold rule**.

In [ ]:
import glob
import os
import numpy as np

pattern = (
    "/content/MoMADiff/generation/test_gif/"
    "a_person_walks_forward_and_makes_a_right_turn/*.npy"
)
files = glob.glob(pattern)
if not files:
    raise FileNotFoundError("WALK → TURN motion not found.")
motion_path = max(files, key=os.path.getmtime)
print("Selected:", motion_path)

motion = np.load(motion_path)

print("Motion shape:", motion.shape)


# ------------------------------------------------------------
# 2. HumanML3D joint indices
# ------------------------------------------------------------

ROOT = 0
LEFT_HIP = 1
RIGHT_HIP = 2
LEFT_SHOULDER = 16
RIGHT_SHOULDER = 17


# ------------------------------------------------------------
# 3. Root trajectory (horizontal X-Z plane)
# ------------------------------------------------------------

root_xz = motion[:, ROOT, [0, 2]]

root_step = np.linalg.norm(
    np.diff(root_xz, axis=0),
    axis=1
)


# ------------------------------------------------------------
# 4. Estimate body forward direction
# ------------------------------------------------------------

# Left-right body axis from hips
hip_lr = (
    motion[:, RIGHT_HIP, [0, 2]]
    - motion[:, LEFT_HIP, [0, 2]]
)

# Left-right body axis from shoulders
shoulder_lr = (
    motion[:, RIGHT_SHOULDER, [0, 2]]
    - motion[:, LEFT_SHOULDER, [0, 2]]
)

body_lr = hip_lr + shoulder_lr


# IMPORTANT:
# Corrected forward direction after calibration
# using "A person walks forward."
forward = np.stack(
    [
        body_lr[:, 1],
        -body_lr[:, 0]
    ],
    axis=1
)

# Normalize
forward /= (
    np.linalg.norm(
        forward,
        axis=1,
        keepdims=True
    )
    + 1e-8
)


# ------------------------------------------------------------
# 5. Calculate body heading
# ------------------------------------------------------------

heading = np.unwrap(
    np.arctan2(
        forward[:, 1],
        forward[:, 0]
    )
)

heading_deg = np.degrees(heading)

relative_heading = (
    heading_deg
    - heading_deg[0]
)


# ------------------------------------------------------------
# 6. Detect TURN onset
# ------------------------------------------------------------

TURN_THRESHOLD_DEG = 15.0

turn_frames = np.where(
    np.abs(relative_heading)
    >= TURN_THRESHOLD_DEG
)[0]

if len(turn_frames) > 0:
    turn_start = int(turn_frames[0])
else:
    turn_start = None


# ------------------------------------------------------------
# 7. Analyze movement BEFORE turn
# ------------------------------------------------------------

initial_forward = forward[0].copy()

initial_forward /= (
    np.linalg.norm(initial_forward)
    + 1e-8
)

initial_side = np.array([
    -initial_forward[1],
    initial_forward[0]
])


if turn_start is not None:

    displacement_before_turn = (
        root_xz[turn_start]
        - root_xz[0]
    )

    total_before_turn = np.linalg.norm(
        displacement_before_turn
    )

    forward_displacement = np.dot(
        displacement_before_turn,
        initial_forward
    )

    sideways_displacement = np.dot(
        displacement_before_turn,
        initial_side
    )

    forward_proportion = (
        forward_displacement
        / (total_before_turn + 1e-8)
    )

else:

    total_before_turn = None
    forward_displacement = None
    sideways_displacement = None
    forward_proportion = None


# ------------------------------------------------------------
# 8. Overall motion statistics
# ------------------------------------------------------------

total_root_displacement = np.linalg.norm(
    root_xz[-1]
    - root_xz[0]
)

total_heading_change = (
    relative_heading[-1]
)

max_heading_change = np.max(
    np.abs(
        np.diff(heading_deg)
    )
)


# ------------------------------------------------------------
# 9. Display evaluator evidence
# ------------------------------------------------------------

print("\n======================================")
print("ORDER EVALUATOR — WALK → TURN")
print("======================================")

print("\n--- Motion ---")

print(
    "Frames:",
    len(motion)
)

print(
    "Total root displacement:",
    round(total_root_displacement, 3)
)

print(
    "Mean movement/frame:",
    round(np.mean(root_step), 4)
)


print("\n--- TURN Event ---")

print(
    "Total heading change:",
    round(total_heading_change, 1),
    "degrees"
)

print(
    "Maximum frame heading change:",
    round(max_heading_change, 1),
    "degrees"
)

print(
    "Turn start frame:",
    turn_start
)


if turn_start is not None:

    print(
        "Turn starts at:",
        round(
            turn_start
            / len(motion)
            * 100,
            1
        ),
        "% of motion"
    )


print("\n--- PRE-TURN WALK ---")

if turn_start is not None:

    print(
        "Total displacement before turn:",
        round(total_before_turn, 3)
    )

    print(
        "Forward displacement:",
        round(forward_displacement, 3)
    )

    print(
        "Sideways displacement:",
        round(sideways_displacement, 3)
    )

    print(
        "Forward proportion:",
        round(forward_proportion, 3)
    )

else:

    print("No turn event detected.")

# ========================================
# Order Evaluator — Step 5
# Inspect heading progression
# ========================================

# Smooth heading to reduce frame-level noise
window = 5

heading_smooth = np.convolve(
    relative_heading,
    np.ones(window) / window,
    mode="same"
)

print("=== HEADING PROGRESSION ===\n")

# Print every 5 frames
for frame in range(0, len(motion), 5):
    print(
        f"Frame {frame:3d}: "
        f"{heading_smooth[frame]:7.2f}°"
    )

print("\nFinal heading:",
      round(relative_heading[-1], 2),
      "degrees")


# ========================================
# Order Evaluator — Step 7
# Window-based Root Trajectory Analysis
# ========================================

WINDOW = 10

# ------------------------------------------------
# 1. Calculate movement direction over each window
# ------------------------------------------------

window_directions = []
window_frames = []

for start in range(0, len(root_xz) - WINDOW):

    end = start + WINDOW

    # Root displacement across the window
    disp = root_xz[end] - root_xz[start]

    distance = np.linalg.norm(disp)

    # Ignore extremely small movements
    if distance < 1e-6:
        continue

    direction = disp / distance

    # Compare with initial body-forward direction
    dot = np.dot(
        direction,
        initial_forward
    )

    dot = np.clip(dot, -1.0, 1.0)

    deviation = np.degrees(
        np.arccos(dot)
    )

    window_directions.append(deviation)

    # Use centre frame of window
    window_frames.append(
        start + WINDOW // 2
    )


window_directions = np.array(window_directions)
window_frames = np.array(window_frames)


# ------------------------------------------------
# 2. Print every ~5 frames
# ------------------------------------------------

print("=== 10-FRAME TRAJECTORY ANALYSIS ===\n")

for i in range(0, len(window_frames), 5):

    print(
        f"Frame {window_frames[i]:3d}: "
        f"trajectory deviation = "
        f"{window_directions[i]:6.2f}°"
    )


# ------------------------------------------------
# 3. Compare early / middle / late motion
# ------------------------------------------------

n = len(window_directions)

early = window_directions[:n // 3]
middle = window_directions[n // 3: 2 * n // 3]
late = window_directions[2 * n // 3:]

print("\n=== PHASE SUMMARY ===")

print(
    "Early mean deviation :",
    round(np.mean(early), 2),
    "degrees"
)

print(
    "Middle mean deviation:",
    round(np.mean(middle), 2),
    "degrees"
)

print(
    "Late mean deviation  :",
    round(np.mean(late), 2),
    "degrees"
)

## 11.2 Candidate Evaluator — Count: N

Detect action-specific geometric events (for example jump apex events), count them, and compare the detected count with the expected count.

**Status:** add the validated group counting rule here once the final detector is agreed.

## 11.3 Candidate Evaluator — Action Omission

Detect whether each required action event is present.

Example: `WALK + JUMP + TURN` expected, but only `WALK + TURN` detected → `JUMP` is omitted.

**Status:** add the validated action-presence detector here once finalized.

# 12. Human Gold Validation

Compare each automatic method with the manually assigned Human Gold Label.

Suggested reporting table:

| Prompt / Requirement | Human Gold | Matching Score | TMR++ | Geometry | Agreement |
|---|---|---:|---:|---|---|
| WALK → TURN | Fail | TBD | TBD | TBD | TBD |
| JUMP → TURN | Fail | TBD | TBD | TBD | TBD |
| WALK → JUMP → TURN | Fail | TBD | TBD | TBD | TBD |

Do not freeze a geometric threshold from a single failure example. Validate candidate rules against the available pilot cases first.

In [ ]:
import numpy as np
import glob
import os

# ==========================================
# Find motion
# ==========================================
files = glob.glob(
    "/content/MoMADiff/generation/test_gif/"
    "a_person_walks_to_the_left/*.npy"
)

print("Found:", len(files))

for i, f in enumerate(files):
    print(i, f)

# 1つだけならこれを使用
motion_path = files[0]

motion = np.load(motion_path)

print("\nMotion:", os.path.basename(motion_path))
print("Shape:", motion.shape)


# ==========================================
# Joint IDs
# ==========================================
LEFT_HIP = 1
RIGHT_HIP = 2
LEFT_SHOULDER = 16
RIGHT_SHOULDER = 17
ROOT = 0


# ==========================================
# Initial body left-right axis
# ==========================================

# Right side - Left side
hip_lr = motion[0, RIGHT_HIP, [0, 2]] - motion[0, LEFT_HIP, [0, 2]]

shoulder_lr = (
    motion[0, RIGHT_SHOULDER, [0, 2]]
    - motion[0, LEFT_SHOULDER, [0, 2]]
)

body_right = hip_lr + shoulder_lr
body_right = body_right / np.linalg.norm(body_right)

# Therefore body-left direction is opposite
body_left = -body_right


# ==========================================
# Root displacement
# ==========================================

start = motion[0, ROOT, [0, 2]]
end   = motion[-1, ROOT, [0, 2]]

displacement = end - start


# ==========================================
# Project displacement onto body L/R axis
# ==========================================

right_amount = np.dot(displacement, body_right)
left_amount  = np.dot(displacement, body_left)


print("\n=== Root displacement ===")
print("Start XZ:", start)
print("End XZ:", end)
print("Displacement:", displacement)

print("\n=== Body-relative direction ===")
print("Movement toward LEFT :", round(left_amount, 4))
print("Movement toward RIGHT:", round(right_amount, 4))


print("\n=== RESULT ===")

if left_amount > 0:
    print("→ Motion moved toward the person's LEFT.")
elif right_amount > 0:
    print("→ Motion moved toward the person's RIGHT.")
else:
    print("→ No clear left/right displacement.")

In [ ]:
import numpy as np

# HumanML3D joint IDs
LEFT_HIP = 1
RIGHT_HIP = 2
LEFT_SHOULDER = 16
RIGHT_SHOULDER = 17

# 最初のフレーム
f = 0

print("=== HIP ===")
print("Left hip  (1) XZ :", motion[f, LEFT_HIP, [0, 2]])
print("Right hip (2) XZ :", motion[f, RIGHT_HIP, [0, 2]])

print("\n=== SHOULDER ===")
print("Left shoulder  (16) XZ :", motion[f, LEFT_SHOULDER, [0, 2]])
print("Right shoulder (17) XZ :", motion[f, RIGHT_SHOULDER, [0, 2]])

# Left - Right のベクトル
hip_to_left = (
    motion[f, LEFT_HIP, [0, 2]]
    - motion[f, RIGHT_HIP, [0, 2]]
)

shoulder_to_left = (
    motion[f, LEFT_SHOULDER, [0, 2]]
    - motion[f, RIGHT_SHOULDER, [0, 2]]
)

left_vector = hip_to_left + shoulder_to_left

print("\n=== Anatomical RIGHT → LEFT vector ===")
print("Hip R→L      :", hip_to_left)
print("Shoulder R→L :", shoulder_to_left)
print("Combined     :", left_vector)

print("\n=== X component ===")

if left_vector[0] > 0:
    print("→ Anatomical LEFT points toward +X")
else:
    print("→ Anatomical LEFT points toward -X")
